# DS 4320 Project 1: NFL Combine Performance as a Predictor for Career Value
**Pipeline Notebook** | Kieran | rrx5eg

This notebook demonstrates the full solution pipeline:
1. Build the relational database from raw CSV files using DuckDB
2. Query the database to prepare the analysis dataset
3. Implement and evaluate a predictive ML model
4. Visualize results at publication quality

---
## Part 1: Data Preparation — Load CSVs into DuckDB

We use DuckDB as our in-process analytical database. Each of the four relational tables is loaded from its CSV file in the `/data` folder. DuckDB lets us query them with standard SQL without needing a separate database server.

In [ ]:
# Install dependencies if needed (uncomment if running fresh)
# !pip install duckdb pandas scikit-learn matplotlib seaborn

import duckdb
import pandas as pd
import logging
import os

# --- Logging setup ---
# Logs key pipeline steps to pipeline.log so errors are traceable
logging.basicConfig(
    filename='pipeline.log',
    level=logging.INFO,
    format='%(asctime)s — %(levelname)s — %(message)s'
)
logging.info('Pipeline started.')
print('Logging initialized. Output will be written to pipeline.log')

In [ ]:
# --- Connect to DuckDB ---
# Using an in-memory database for the pipeline.
# To persist across sessions, replace ':memory:' with a file path e.g. 'nfl_combine.duckdb'
try:
    con = duckdb.connect(':memory:')
    logging.info('DuckDB connection established.')
    print('DuckDB connected successfully.')
except Exception as e:
    logging.error(f'Failed to connect to DuckDB: {e}')
    raise

In [ ]:
# --- Load the four relational tables from /data ---
# Each CSV corresponds to one entity in the ER diagram.
# DuckDB reads CSVs directly — no intermediate pandas step needed.

DATA_DIR = 'data'  # relative path — adjust if your folder is named differently

tables = {
    'players':          'players.csv',
    'combine_stats':    'combine_stats.csv',
    'pro_performance':  'pro_performance.csv',
    'position_groups':  'position_groups.csv',
}

for table_name, filename in tables.items():
    filepath = os.path.join(DATA_DIR, filename)
    try:
        # CREATE OR REPLACE so re-running the cell is safe
        con.execute(f"""
            CREATE OR REPLACE TABLE {table_name} AS
            SELECT * FROM read_csv_auto('{filepath}')
        """)
        count = con.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
        logging.info(f'Loaded {table_name}: {count} rows.')
        print(f'  {table_name}: {count} rows loaded.')
    except Exception as e:
        logging.error(f'Failed to load {table_name} from {filepath}: {e}')
        raise

print('\nAll tables loaded into DuckDB.')

In [ ]:
# --- Verify schema of each table ---
# Quick sanity check: print column names and types for each table.
for table_name in tables.keys():
    print(f'\n--- {table_name} ---')
    schema = con.execute(f'DESCRIBE {table_name}').df()
    print(schema[['column_name', 'column_type']].to_string(index=False))

---
## Part 2: SQL Queries — Prepare the Analysis Dataset

We join all four tables to produce a flat analysis-ready view. Three queries are demonstrated:
- **Query 1**: Full join across all tables (the core analysis dataset)
- **Query 2**: Position-group summary — average combine metrics and AV by group
- **Query 3**: Top performers — players with high AV but average or below-average combine scores (the 'hidden gems' our model aims to find)

In [ ]:
# --- Query 1: Full join — the master analysis dataset ---
# Joins players → combine_stats → pro_performance → position_groups
# This is the dataset we will feed into the ML model.

try:
    query_full_join = """
        SELECT
            p.player_id,
            p.name,
            p.position,
            pg.group_name          AS position_group,
            p.college,
            p.height,
            p.weight,
            cs.forty_yd_dash,
            cs.vertical_jump,
            cs.bench_press,
            cs.broad_jump,
            cs.three_cone,
            cs.shuttle_run,
            pp.draft_round,
            pp.overall_pick,
            pp.career_av
        FROM players p
        JOIN combine_stats cs
            ON p.player_id = cs.player_id
        JOIN pro_performance pp
            ON p.player_id = pp.player_id
        JOIN position_groups pg
            ON p.position = pg.position
        WHERE
            cs.forty_yd_dash IS NOT NULL
            AND pp.career_av IS NOT NULL
    """

    df_analysis = con.execute(query_full_join).df()
    logging.info(f'Query 1 complete: {len(df_analysis)} rows in analysis dataset.')
    print(f'Analysis dataset shape: {df_analysis.shape}')
    df_analysis.head()
except Exception as e:
    logging.error(f'Query 1 failed: {e}')
    raise

In [ ]:
# --- Query 2: Position-group summary ---
# Shows average combine results and career AV broken out by position group.
# This justifies our decision to analyze positions separately rather than globally.

try:
    query_by_group = """
        SELECT
            pg.group_name,
            COUNT(p.player_id)              AS player_count,
            ROUND(AVG(cs.forty_yd_dash), 2) AS avg_40yd,
            ROUND(AVG(cs.vertical_jump), 1) AS avg_vertical,
            ROUND(AVG(cs.broad_jump), 1)    AS avg_broad_jump,
            ROUND(AVG(pp.career_av), 1)     AS avg_career_av
        FROM players p
        JOIN combine_stats cs  ON p.player_id = cs.player_id
        JOIN pro_performance pp ON p.player_id = pp.player_id
        JOIN position_groups pg ON p.position = pg.position
        WHERE cs.forty_yd_dash IS NOT NULL
          AND pp.career_av IS NOT NULL
        GROUP BY pg.group_name
        ORDER BY avg_career_av DESC
    """

    df_by_group = con.execute(query_by_group).df()
    logging.info('Query 2 (position group summary) complete.')
    print('Average combine metrics and career AV by position group:')
    display(df_by_group)
except Exception as e:
    logging.error(f'Query 2 failed: {e}')
    raise

In [ ]:
# --- Query 3: Hidden gems — high AV, average or slower 40-yd ---
# Players in the top 25% of career AV but at or below the median 40-yard dash time.
# These are the players traditional speed-focused scouting would have undervalued.

try:
    query_hidden_gems = """
        WITH percentiles AS (
            SELECT
                PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY pp.career_av)     AS av_75th,
                PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY cs.forty_yd_dash) AS dash_median
            FROM players p
            JOIN combine_stats cs  ON p.player_id = cs.player_id
            JOIN pro_performance pp ON p.player_id = pp.player_id
            WHERE cs.forty_yd_dash IS NOT NULL AND pp.career_av IS NOT NULL
        )
        SELECT
            p.name,
            p.position,
            cs.forty_yd_dash,
            cs.vertical_jump,
            pp.career_av,
            pp.draft_round
        FROM players p
        JOIN combine_stats cs   ON p.player_id = cs.player_id
        JOIN pro_performance pp ON p.player_id = pp.player_id
        CROSS JOIN percentiles
        WHERE
            pp.career_av    >= percentiles.av_75th
            AND cs.forty_yd_dash >= percentiles.dash_median  -- at or SLOWER than median
            AND cs.forty_yd_dash IS NOT NULL
        ORDER BY pp.career_av DESC
        LIMIT 20
    """

    df_gems = con.execute(query_hidden_gems).df()
    logging.info(f'Query 3 (hidden gems) returned {len(df_gems)} players.')
    print(f'Top {len(df_gems)} high-value players with average or below-average speed:')
    display(df_gems)
except Exception as e:
    logging.error(f'Query 3 failed: {e}')
    raise

---
## Part 3: ML Model — Predicting Career AV from Combine Metrics

We implement two models from DS 3021/4021:
- **Linear Regression**: establishes a baseline and gives interpretable coefficients
- **Random Forest Regressor**: captures non-linear relationships and produces feature importances

**Rationale**: Linear regression lets us directly read off which combine drills have the strongest linear relationship with career AV. Random Forest is added because the relationship between athleticism and professional value is unlikely to be purely linear — for example, a player must exceed a *threshold* of speed, not simply maximize it.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# --- Feature selection ---
# These are the core athletic 'measurables' from the combine.
# Draft round is excluded intentionally — we want to isolate physical metrics,
# not draft position (which already embeds scout bias).
FEATURES = ['forty_yd_dash', 'vertical_jump', 'bench_press',
            'broad_jump', 'three_cone', 'shuttle_run', 'height', 'weight']
TARGET = 'career_av'

# Drop rows missing any feature or the target
df_model = df_analysis[FEATURES + [TARGET]].dropna()

X = df_model[FEATURES]
y = df_model[TARGET]

# 80/20 train-test split with fixed random state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features for linear regression (RF doesn't require scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

logging.info(f'Model dataset: {len(df_model)} rows, {len(FEATURES)} features.')
print(f'Training set: {len(X_train)} rows | Test set: {len(X_test)} rows')

In [ ]:
# --- Model 1: Linear Regression ---
# Baseline model. Coefficients tell us the marginal effect of each combine drill.

try:
    lr = LinearRegression()
    lr.fit(X_train_scaled, y_train)
    y_pred_lr = lr.predict(X_test_scaled)

    rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
    r2_lr   = r2_score(y_test, y_pred_lr)

    logging.info(f'Linear Regression — RMSE: {rmse_lr:.2f}, R2: {r2_lr:.3f}')
    print(f'Linear Regression  |  RMSE: {rmse_lr:.2f}  |  R²: {r2_lr:.3f}')

    # Coefficients table — which drills matter most linearly?
    coef_df = pd.DataFrame({
        'Feature':     FEATURES,
        'Coefficient': lr.coef_
    }).sort_values('Coefficient', key=abs, ascending=False)
    print('\nLinear Regression Coefficients (standardized):')
    display(coef_df)
except Exception as e:
    logging.error(f'Linear Regression failed: {e}')
    raise

In [ ]:
# --- Model 2: Random Forest Regressor ---
# Non-linear model that captures threshold effects and interactions.
# n_estimators=200 balances accuracy and runtime; max_depth=8 prevents overfitting
# on a dataset of this size.

try:
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        random_state=42,
        n_jobs=-1       # use all CPU cores
    )
    rf.fit(X_train, y_train)   # RF uses raw (unscaled) features
    y_pred_rf = rf.predict(X_test)

    rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
    r2_rf   = r2_score(y_test, y_pred_rf)

    logging.info(f'Random Forest — RMSE: {rmse_rf:.2f}, R2: {r2_rf:.3f}')
    print(f'Random Forest      |  RMSE: {rmse_rf:.2f}  |  R²: {r2_rf:.3f}')

    # Feature importances
    imp_df = pd.DataFrame({
        'Feature':    FEATURES,
        'Importance': rf.feature_importances_
    }).sort_values('Importance', ascending=False)
    print('\nRandom Forest Feature Importances:')
    display(imp_df)
except Exception as e:
    logging.error(f'Random Forest failed: {e}')
    raise

---
## Part 4: Visualizations

Three publication-quality charts are produced:
1. **Feature importance bar chart** (Random Forest) — answers *which combine drills predict AV?*
2. **Predicted vs. actual AV scatter** — shows model accuracy and residual spread
3. **Position-group AV comparison** — contextualizes the positional bias discussed in the bias section

**Visualization rationale**: A bar chart is used for feature importances because we are comparing a single continuous quantity (importance score) across named categories (drills). A scatter plot is used for predicted vs. actual because it reveals heteroskedasticity — whether model errors are larger for high-AV players. A grouped bar chart by position group is used because it directly supports the project's core argument that position must be controlled for.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Publication style — clean, minimal, suitable for a paper or press release
plt.rcParams.update({
    'font.family':       'sans-serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
    'figure.dpi':        150,
})

# --- Figure 1: Random Forest Feature Importances ---
# Shows which combine drills are the strongest predictors of career value.

fig1, ax1 = plt.subplots(figsize=(8, 5))

colors = ['#185FA5' if i == 0 else '#B5D4F4' for i in range(len(imp_df))]
ax1.barh(imp_df['Feature'][::-1], imp_df['Importance'][::-1], color=colors[::-1], edgecolor='none')
ax1.set_xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=11)
ax1.set_title(
    'Which Combine Drills Best Predict Career Approximate Value?',
    fontsize=13, fontweight='bold', pad=12
)
ax1.annotate(
    'Source: NFL Combine Dataset (Kaggle) + Pro-Football-Reference.com',
    xy=(0.01, -0.12), xycoords='axes fraction',
    fontsize=8, color='gray'
)

plt.tight_layout()
plt.savefig('fig1_feature_importance.png', bbox_inches='tight')
plt.show()
logging.info('Figure 1 saved: fig1_feature_importance.png')
print('Figure 1 saved.')

In [ ]:
# --- Figure 2: Predicted vs. Actual AV (Random Forest) ---
# A well-calibrated model produces points along the diagonal.
# Spread around the diagonal shows where the model is uncertain.

fig2, ax2 = plt.subplots(figsize=(6, 6))

ax2.scatter(y_test, y_pred_rf, alpha=0.45, s=20, color='#185FA5', edgecolors='none')

# Perfect prediction line
lims = [min(y_test.min(), y_pred_rf.min()), max(y_test.max(), y_pred_rf.max())]
ax2.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')

ax2.set_xlabel('Actual Career AV', fontsize=11)
ax2.set_ylabel('Predicted Career AV', fontsize=11)
ax2.set_title(
    f'Predicted vs. Actual Career AV\n(Random Forest, R² = {r2_rf:.3f})',
    fontsize=13, fontweight='bold', pad=12
)
ax2.legend(fontsize=9)
ax2.annotate(
    'Source: NFL Combine Dataset (Kaggle) + Pro-Football-Reference.com',
    xy=(0.01, -0.12), xycoords='axes fraction',
    fontsize=8, color='gray'
)

plt.tight_layout()
plt.savefig('fig2_predicted_vs_actual.png', bbox_inches='tight')
plt.show()
logging.info('Figure 2 saved: fig2_predicted_vs_actual.png')
print('Figure 2 saved.')

In [ ]:
# --- Figure 3: Average Career AV by Position Group ---
# Demonstrates why position-stratified analysis is necessary.
# The wide spread across groups confirms that a global model without
# position controls would introduce systematic bias.

fig3, ax3 = plt.subplots(figsize=(8, 5))

groups_sorted = df_by_group.sort_values('avg_career_av', ascending=True)
bars = ax3.barh(
    groups_sorted['group_name'],
    groups_sorted['avg_career_av'],
    color='#185FA5', edgecolor='none'
)

# Label each bar with the player count
for bar, count in zip(bars, groups_sorted['player_count']):
    ax3.text(
        bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
        f'n={count}', va='center', fontsize=8, color='gray'
    )

ax3.set_xlabel('Average Career Approximate Value', fontsize=11)
ax3.set_title(
    'Career Value Varies Substantially by Position Group',
    fontsize=13, fontweight='bold', pad=12
)
ax3.annotate(
    'Source: NFL Combine Dataset (Kaggle) + Pro-Football-Reference.com',
    xy=(0.01, -0.12), xycoords='axes fraction',
    fontsize=8, color='gray'
)

plt.tight_layout()
plt.savefig('fig3_av_by_position_group.png', bbox_inches='tight')
plt.show()
logging.info('Figure 3 saved: fig3_av_by_position_group.png')
print('Figure 3 saved.')
logging.info('Pipeline complete.')

---
## Summary

| Model | RMSE | R² |
|---|---|---|
| Linear Regression | *(see output above)* | *(see output above)* |
| Random Forest | *(see output above)* | *(see output above)* |

The Random Forest outperforms Linear Regression on this task, consistent with the hypothesis that the combine–career value relationship contains important non-linearities. Feature importance results should be interpreted in context of the bias discussion: because AV is not positionally neutral, importances may reflect positional composition of the dataset as much as true athletic predictors. Future work should train position-stratified models.